# ResNet34 on manually labeled ERP image patterns, 128x128 Gaussian

This notebook syncs local Label Studio annotations, builds fixed-trial augmented ERP images from the labeled sources, and runs a 5-fold cross-validation with an ImageNet-pretrained Metalhead ResNet34. The prediction target remains binary: `class` versus `no_class`.

Implementation details are in `resnet34_labeled_erp_cv_128_resize.jl` so the experiment can also be run from the terminal. Preprocessing is `sort -> zscore_timepoints -> Gaussian smoothing -> resize 128x128`, using the same utility pipeline and low-pass settings as the ResNet18 notebook.

In [ ]:
import Pkg

function find_repo_root(start::AbstractString = pwd())
    candidate = start
    for _ in 1:8
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
        candidate = dirname(candidate)
    end
    error("Could not locate repo root from ", start)
end

REPO_ROOT = find_repo_root()
WEEK21 = joinpath(REPO_ROOT, "notebooks", "week_21")
OUTPUT_DIR = joinpath(WEEK21, "outputs", "resnet34_labeled_erp_cv_128_resize")
PLOTS_DIR = joinpath(OUTPUT_DIR, "plots")

Pkg.activate(joinpath(REPO_ROOT, "notebooks", "model_test"))
using CSV, DataFrames, JSON3, CairoMakie
REPO_ROOT

## 1. Sync Label Studio labels

This step reads the local Label Studio SQLite DB and updates the tracking CSVs, including the most recent follow-up projects.

In [ ]:
run(Cmd(`python3 $(joinpath(WEEK21, "update_labelstudio_annotation_tracking.py"))`; dir = REPO_ROOT))

## 2. Run ResNet34 cross-validation

The Julia script uses `WEEK21_TARGET_TRIALS` as the fixed trial count for every augmented ERP image; the default is 150. Positive pattern rows keep all round-robin mod-split chunks plus a filled remainder chunk when a remainder exists; `no_class` rows are augmented the same way but only one deterministic chunk is kept for class balance. Folds are stratified by the seven manual labels, so pattern types are distributed across folds. Metalhead documents `ResNet(depth; pretrain, inchannels, nclasses)` with supported depth `34`; this script loads the ImageNet ResNet34 weights and projects the first convolution to one input channel. The ERP image preprocessing uses `gaussian_reference` from the shared utils with the project-wide low-pass factor and kernel size.

In [ ]:
# Optional overrides before running:
# ENV["WEEK21_RESNET34_EPOCHS"] = "8"
# ENV["WEEK21_RESNET34_BATCHSIZE_GPU"] = "32"
# ENV["WEEK21_NO_CLASS_CHUNKS_PER_ORIGIN"] = "1"
get!(ENV, "WEEK21_TARGET_TRIALS", "150")
get!(ENV, "WEEK21_RESNET34_TARGET_HEIGHT", "128")
get!(ENV, "WEEK21_RESNET34_TARGET_WIDTH", "128")

run(Cmd(`julia --project=notebooks/model_test $(joinpath(WEEK21, "resnet34_labeled_erp_cv_128_resize.jl"))`; dir = REPO_ROOT))

## 3. Inspect outputs

In [ ]:
metrics_summary       = CSV.read(joinpath(OUTPUT_DIR, "metrics_summary.csv"), DataFrame)
fold_metrics          = CSV.read(joinpath(OUTPUT_DIR, "fold_metrics.csv"), DataFrame)
fold_classes          = CSV.read(joinpath(OUTPUT_DIR, "fold_distribution_pattern_class.csv"), DataFrame)
sample_plan           = CSV.read(joinpath(OUTPUT_DIR, "sample_plan.csv"), DataFrame)
augmented_summary     = CSV.read(joinpath(OUTPUT_DIR, "augmented_label_summary.csv"), DataFrame)
train_history         = CSV.read(joinpath(OUTPUT_DIR, "train_history.csv"), DataFrame)
predictions           = CSV.read(joinpath(OUTPUT_DIR, "validation_predictions.csv"), DataFrame)
run_config            = JSON3.read(read(joinpath(OUTPUT_DIR, "run_config.json"), String))

config_subset = (
    model_name              = run_config.model_name,
    target_trials           = run_config.target_trials,
    target_size             = run_config.target_size,
    preprocessing_pipeline  = run_config.preprocessing_pipeline,
    n_labeled_rows_used     = run_config.n_labeled_rows_used,
    n_augmented_images      = run_config.n_augmented_images,
    k_folds                 = run_config.k_folds,
    nepochs                 = run_config.nepochs,
)
metrics_summary, config_subset

In [ ]:
trial_counts = Dict{Int, Int}()
for n in sample_plan.n_trials
    trial_counts[Int(n)] = get(trial_counts, Int(n), 0) + 1
end
@assert trial_counts == Dict(150 => nrow(sample_plan))

fold_binary = CSV.read(joinpath(OUTPUT_DIR, "fold_distribution_binary.csv"), DataFrame)
first(fold_classes, 12), fold_binary, trial_counts

## 4. Plots

In [ ]:
mkpath(PLOTS_DIR)

function save_show(fig, name)
    path = joinpath(PLOTS_DIR, name)
    save(path, fig)
    return path
end

metric_labels = [
    (:val_accuracy,          "accuracy"),
    (:val_balanced_accuracy, "balanced accuracy"),
    (:val_macro_f1,          "macro F1"),
]

fig = Figure(size = (900, 480))
ax = Axis(fig[1, 1];
    title = "ResNet34 128x128 Gaussian CV metrics",
    xlabel = "fold",
    ylabel = "validation score",
)
folds = Int.(fold_metrics.fold)
for (col, lbl) in metric_labels
    scatterlines!(ax, folds, Float64.(fold_metrics[!, col]); marker = :circle, label = lbl)
end
ax.xticks = (folds, string.(folds))
ylims!(ax, 0, 1)
axislegend(ax; position = :rb)
save_show(fig, "validation_metrics_by_fold.png")
fig

In [ ]:
fig = Figure(size = (900, 480))
ax = Axis(fig[1, 1];
    title = "Training loss by fold",
    xlabel = "epoch",
    ylabel = "training loss",
)
for f in sort(unique(Int.(train_history.fold)))
    sub = sort(train_history[Int.(train_history.fold) .== f, :], :epoch)
    scatterlines!(ax, Int.(sub.epoch), Float64.(sub.avg_loss);
        marker = :circle, label = "fold $(f)")
end
axislegend(ax; nbanks = 3, position = :rt)
save_show(fig, "training_loss_by_fold.png")
fig

In [ ]:
class_order = ["no_class", "sigmoid", "one_sided_fan", "two_sided_fan",
               "diverging_bar", "hourglass", "tilted_bar"]
fold_order = sort(unique(Int.(fold_classes.fold)))
counts = Dict{Tuple{Int, String}, Int}()
for r in eachrow(fold_classes)
    counts[(Int(r.fold), String(r.erp_class))] = Int(r.count)
end

fig = Figure(size = (1100, 540))
ax = Axis(fig[1, 1];
    title = "Manual pattern class distribution per fold",
    xlabel = "fold",
    ylabel = "augmented images",
    xticks = (fold_order, string.(fold_order)),
)
bottom = zeros(Float64, length(fold_order))
for cls in class_order
    vals = Float64[get(counts, (f, cls), 0) for f in fold_order]
    barplot!(ax, Float64.(fold_order), vals; offset = copy(bottom), label = cls)
    bottom .+= vals
end
Legend(fig[1, 2], ax; framevisible = false)
save_show(fig, "fold_pattern_class_distribution.png")
fig

In [ ]:
label_names = ["no_class", "class"]
cm = zeros(Int, 2, 2)
for r in eachrow(predictions)
    cm[Int(r.true_binary_label) + 1, Int(r.predicted_binary_label) + 1] += 1
end

fig = Figure(size = (560, 480))
ax = Axis(fig[1, 1];
    title = "Validation confusion matrix",
    xlabel = "predicted",
    ylabel = "true",
    xticks = (1:2, label_names),
    yticks = (1:2, label_names),
    yreversed = true,
)
hm = heatmap!(ax, 1:2, 1:2, transpose(cm); colormap = :Blues)
for i in 1:2, j in 1:2
    text!(ax, Float64(j), Float64(i); text = string(cm[i, j]),
        align = (:center, :center), color = :black)
end
Colorbar(fig[1, 2], hm)
save_show(fig, "validation_confusion_matrix.png")
fig

In [ ]:
summary_counts = Dict(String(r.erp_class) => Int(r.count) for r in eachrow(augmented_summary))
vals = Float64[get(summary_counts, cls, 0) for cls in class_order]

fig = Figure(size = (1100, 480))
ax = Axis(fig[1, 1];
    title = "Augmented label distribution",
    ylabel = "augmented images",
    xticks = (1:length(class_order), class_order),
    xticklabelrotation = pi / 6,
)
barplot!(ax, 1:length(class_order), vals)
save_show(fig, "augmented_label_distribution.png")
fig